## Sanskrit Machine Learning!

 Using NLP to learn embeddings from Vedic texts and explore conceptual similarity between verses, deities, and philosophical ideas (Dharma, Rta, Atman, Brahman).


### Loading the Dataset:
#### This is for the Kaggle Dataset!


Imports needed for Kaggle

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [ ]:
# This is the CSV file INSIDE the Kaggle dataset
file_path = "complete_rigveda_all_mandalas.json"

# Load the dataset as a pandas DataFrame
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "varunrajuvangar/rigved-all-sukta-verses-and-meaning-dataset",
    file_path,
)

# print("First 5 records:")
# print(df.head())

print("\nColumns:")
print(df.columns)

In [ ]:
# Display the first Sukta data to verify content and structure for cleaning
import pprint
first_sukta_data = df.iloc[0,0]
print("\nFirst Sukta Data:")
pprint.pprint(first_sukta_data)


In [ ]:
import pandas as pd

every_verse = []

for mandala in df.columns:
    for sukta in df.index:
        curr_cell = df.at[sukta, mandala]

        if not isinstance(curr_cell, list):
            continue

        for verse in curr_cell:
            try:
                sanskrit_verse = verse['samhita']['devanagari']['text']
                display_sanskrit = verse['sanskrit_wisdomlib']
                eng_translation = verse['translation']
                verse_num = verse['rik_number']
                
                every_verse.append({
                    'mandala': mandala,
                    'sukta': sukta,
                    'verse_num': verse_num,
                    'sanskrit_verse': sanskrit_verse,
                    'display_sanskrit': display_sanskrit,
                    'english_translation': eng_translation
                })
            
            except KeyError as e:
                print(f"KeyError for mandala {mandala}, sukta {sukta}, verse {verse.get('rik_number', '?')}: {e}")

cleaned_df = pd.DataFrame(every_verse)
print("\nCleaned DataFrame head:")
print(cleaned_df.head())

## Embeddings for English Translations

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded successfully!")

In [ ]:
print("Generating embeddings...")
embeddings = model.encode(
    cleaned_df['english_translation'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"Embeddings shape: {embeddings.shape}")
cleaned_df['embedding'] = list(embeddings)

In [ ]:
# 1. Look at a single verse embedding
print("Single verse embedding (first 10 dimensions):")
print(embeddings[0][:10])
print(f"\nFull embedding shape for one verse: {embeddings[0].shape}")

# 2. See embeddings for first 3 verses
print("\nFirst 3 verse embeddings:")
print(embeddings[:3])

# 3. Compare two verse embeddings side-by-side
print("\nComparing verse 0 and verse 1:")
print(f"Verse 0 embedding: {embeddings[0][:5]}...")
print(f"Verse 1 embedding: {embeddings[1][:5]}...")

In [ ]:
# Preparation (Do this once):
# You have your DataFrame cleaned_df.
# You have a column embeddings which contains the vectors for every verse.
# Crucial Step: Extract all those individual vectors from the DataFrame and stack them into one big block (a matrix). This makes the math fast.
verse_matrix = np.vstack(cleaned_df['embedding'].values)
print(f"\nVerse matrix shape: {verse_matrix.shape}")
print(verse_matrix[:2, :5])  

# The Search Function (Run this every time you search):
# Input: Take a text string (the "Query") from the user.
# Vectorize: Feed that text string into your SentenceTransformer model. This spits out a single vector (list of numbers).
# Math: Compare that Single Query Vector against the Big Matrix of Verse Vectors.
# Score: The result will be a list of 10,000 scores (between -1 and 1).
# Assign: Paste these scores back into your DataFrame as a new temporary column called "similarity_score".
# Sort: Sort the DataFrame so the rows with the highest "similarity_score" are at the top.
# Slice: Cut off the top 5 or 10 rows.
# Output: Print the English translation and Sanskrit text for those top rows.

In [ ]:
def search_verses(query, top_k):
    pass

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

X = np.array([[0, 0, 0], [1, 1, 1]])
Y = np.array([[1, 0, 0], [1, 1, 0]])

similarity_scores = cosine_similarity(X, Y)
print("Similarity Scores:")
print(similarity_scores)